Build A Basic Chatbot With Langgraph(GRAPH API)

In [7]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [9]:
class State(TypedDict):
    # Messages have the type "list". The 'add_messages' function in the annotation defines how this state key should be updated.
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

graph_builder= StateGraph(State)   


In [10]:
graph_builder

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm=ChatGroq(model="qwen/qwen3.8-27b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'Qwen3.8 27B', 'status': 'beta', 'release_date': '2026-08-14', 'last_updated': '2026-08-14', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000014863CBAFD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000014863CBB9D0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [20]:
## node functionality
def chatbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

In [22]:
graph_builder=StateGraph(State)

##creating node
graph_builder.add_node("chatbot", chatbot)

##creating edges
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

##compile the graph
graph=graph_builder.compile()


In [23]:
graph.invoke({"messages":"Hi"})

{'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='b1ca4654-7c19-4108-bea9-d3d14436c112'),
  AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_time': 0.019190192, 'completion_tokens_details': None, 'prompt_time': 0.000601701, 'prompt_tokens_details': None, 'queue_time': 0.054131809, 'total_time': 0.019791893}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1fe9c-1cf0-767e-809e-f272dc29a700-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23})]}